## Initializing


### Importing packages


In [2]:
import numpy as np
import polars as pl
from pathlib import Path

### Setting up global variables


In [3]:
DATA_DIR = Path("../Dataset Round 01/Data")
OUTPUT_DIR = Path("../Dataset Round 01/Cleaned")

SEGMENTATION_DATASET = "2017Segmentation3685case.csv"
BRAND_IMAGE_DATASET = "Brand_Image.csv"
BRAND_HEALTH_DATASET = "Brandhealth.csv"
COMPANION_DATASET = "Companion.csv"
COMPETITOR_DATASET = "Competitor database_xlnm#_FilterDatabase.csv"
DAYOFWEEK_DATASET = "Dayofweek.csv"
DAYPART_DATASET = "Daypart.csv"
NEEDSTATE_DATASET = "NeedstateDayDaypart.csv"
SA_DATASET = "SA#var.csv"

## Cleaning data


### Segmentation dataset


#### Read the dataset


In [4]:
df = pl.read_csv(DATA_DIR / SEGMENTATION_DATASET, separator=";")
df

ID,Segmentation,Visit,Spending,Brand,PPA
i64,str,i64,i64,str,i64
92316,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30
96307,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30
105678,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30
106554,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30
106555,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30
…,…,…,…,…,…
139086,"""Seg.01 - Mass (<VND 25K)""",30,300,"""Street""",10
139137,"""Seg.01 - Mass (<VND 25K)""",30,300,"""Street""",10
139183,"""Seg.01 - Mass (<VND 25K)""",30,300,"""Street""",10


#### Check for abnormality


Check for null values


In [5]:
df.null_count()

ID,Segmentation,Visit,Spending,Brand,PPA
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


Check if $\cfrac{\text{Spending}}{\text{Visit}} = \text{PPA}$.


In [14]:
df.filter(pl.col("Spending") / pl.col("Visit") != pl.col("PPA"))

ID,Segmentation,Visit,Spending,Brand,PPA,Segmentation ID,Average Min,Average Max
i64,str,i64,i64,str,i64,i64,f64,f64


Check if the segmentation is in correct format


In [6]:
df.select("Segmentation").unique()

Segmentation
str
"""Seg.04 - Super Premium (VND 10…"
"""Seg.02 - Mass Asp (VND 25K - V…"
"""Seg.01 - Mass (<VND 25K)"""
"""Seg.03 - Premium (VND 60K - VN…"


Split the segmentation column into a new dataset with four columns:

- Segmentation ID
- Segmentation name
- Average spending range (min, and max).


In [7]:
segmentation_df = pl.DataFrame(
    {
        "Segmentation ID": [1, 2, 3, 4],
        "Segmentation Name": df.select("Segmentation")
        .unique()
        .sort("Segmentation")
        .to_series(),
        "Average Min": [-np.inf, 25, 60, 100],
        "Average Max": [25, 60, 100, np.inf],
    },
    strict=False,
)

df = df.join(segmentation_df, left_on="Segmentation", right_on="Segmentation Name")
df

ID,Segmentation,Visit,Spending,Brand,PPA,Segmentation ID,Average Min,Average Max
i64,str,i64,i64,str,i64,i64,f64,f64
92316,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30,2,25.0,60.0
96307,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30,2,25.0,60.0
105678,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30,2,25.0,60.0
106554,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30,2,25.0,60.0
106555,"""Seg.02 - Mass Asp (VND 25K - V…",4,120,"""Indepentdent""",30,2,25.0,60.0
…,…,…,…,…,…,…,…,…
139086,"""Seg.01 - Mass (<VND 25K)""",30,300,"""Street""",10,1,-inf,25.0
139137,"""Seg.01 - Mass (<VND 25K)""",30,300,"""Street""",10,1,-inf,25.0
139183,"""Seg.01 - Mass (<VND 25K)""",30,300,"""Street""",10,1,-inf,25.0


Check if the PPA is in correct range.


In [18]:
df.filter(
    [pl.col("PPA") >= pl.col("Average Max"), pl.col("PPA") < pl.col("Average Min")]
)

ID,Segmentation,Visit,Spending,Brand,PPA,Segmentation ID,Average Min,Average Max
i64,str,i64,i64,str,i64,i64,f64,f64


We can conclude that no abnormality is found in the dataset.


#### Save the output

- We save both the segmentation dataset and filter out unnecessary columns in `df`, which include:
  - `Segmentation` (use the ID instead).
  - `Average Min and Max` (since they are from the segmentation dataset).
  - `Speding`, since we can derive it using the formula $\text{Spending} = \text{PPA}\cdot\text{Visit}$.


In [22]:
segmentation_df.write_csv(OUTPUT_DIR / "Segmentation Info.csv")
df.select(["ID", "Visit", "PPA", "Segmentation ID", "Brand"]).write_csv(
    OUTPUT_DIR / SEGMENTATION_DATASET
)